## Transformation - Weather data

#### Dataset documentation
* https://mesonet.agron.iastate.edu/request/download.phtml
* https://www.weather.gov/media/asos/aum-toc.pdf

### Transformation checklist
1. [x] Rename columns with names that makes more sense.
2. [x] Make the date / valid column use the proper date type.
4. [x] Deal with missing values.

* The missing values denoted by `M` will be dealt later after joining this weather dataset with the flight dataset.

In [1]:
import sys
from pathlib import Path

project_root = "/notebooks/ML/Flight Delay Prediction Project/ELT"
sys.path.append(str(project_root))

In [2]:
import pandas as pd
from src.transform.weather_data import *
from src.load.weather_data import *

In [ ]:
s3_key = "bronze/Flight_Delay_Prediction_Datasets/weather_data/weather_data.csv"
df = get_lake_dataset(s3_key)

In [4]:
df.head()

,Unnamed: 0,station,valid,tmpf,dwpf,relh,sknt,gust,vsby,skyc1,skyc2,skyc3,wxcodes,p01i,alti
0,0,HTS,2025-01-01 00:51,46.00,37.00,70.64,15.00,25.00,10.00,FEW,BKN,OVC,M,0.01,29.66
1,1,SAN,2025-01-01 00:51,57.00,49.00,74.55,4.00,M,9.00,BKN,M,M,M,0.00,30.05
2,2,SGF,2025-01-01 00:52,36.00,29.00,75.46,11.00,M,10.00,OVC,M,M,M,0.00,30.14
3,3,BTR,2025-01-01 00:53,62.00,39.00,42.56,4.00,M,10.00,CLR,M,M,M,0.00,30.03
4,4,HOU,2025-01-01 00:53,64.00,40.00,41.25,4.00,M,10.00,CLR,M,M,M,0.00,30.07


In [5]:
df.shape

(2694942, 15)

## Rename columns

In [6]:
new_column_names = {
    'valid': 'timestamp', 'tmpf': 'air_temp_f', 'dwpf': 'dew_point_temp_f', 'relh': 'relative_humidity',
    'sknt': 'wind_speed_kts', 'gust': 'wind_gust_kts', 'vsby': 'visibility_miles', 'skyc1': 'sky_l1_coverage',
    'skyc2': 'sky_l2_coverage', 'skyc3': 'sky_l3_coverage', 'wxcodes': 'present_weather_codes', 
    'p01i': '1hr_precipitation_inches', 'alti': 'pressure_altimeter_inches'
}

In [7]:
df_t1 = rename_columns(df, new_column_names)

In [8]:
df_t1.head()

,station,timestamp,air_temp_f,dew_point_temp_f,relative_humidity,wind_speed_kts,wind_gust_kts,visibility_miles,sky_l1_coverage,sky_l2_coverage,sky_l3_coverage,present_weather_codes,1hr_precipitation_inches,pressure_altimeter_inches
0,HTS,2025-01-01 00:51,46.00,37.00,70.64,15.00,25.00,10.00,FEW,BKN,OVC,M,0.01,29.66
1,SAN,2025-01-01 00:51,57.00,49.00,74.55,4.00,M,9.00,BKN,M,M,M,0.00,30.05
2,SGF,2025-01-01 00:52,36.00,29.00,75.46,11.00,M,10.00,OVC,M,M,M,0.00,30.14
3,BTR,2025-01-01 00:53,62.00,39.00,42.56,4.00,M,10.00,CLR,M,M,M,0.00,30.03
4,HOU,2025-01-01 00:53,64.00,40.00,41.25,4.00,M,10.00,CLR,M,M,M,0.00,30.07


In [10]:
df_t1.shape

(2694942, 14)

## Timestamp type correction

In [11]:
df_t2 = timestamp_correction(df_t1, 'timestamp')

In [12]:
df_t2.info()

<class 'pandas.DataFrame'>
RangeIndex: 2694942 entries, 0 to 2694941
Data columns (total 14 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   station                    str           
 1   timestamp                  datetime64[us]
 2   air_temp_f                 str           
 3   dew_point_temp_f           str           
 4   relative_humidity          str           
 5   wind_speed_kts             str           
 6   wind_gust_kts              str           
 7   visibility_miles           str           
 8   sky_l1_coverage            str           
 9   sky_l2_coverage            str           
 10  sky_l3_coverage            str           
 11  present_weather_codes      str           
 12  1hr_precipitation_inches   str           
 13  pressure_altimeter_inches  object        
dtypes: datetime64[us](1), object(1), str(12)
memory usage: 389.3+ MB


In [13]:
df_t2.head()

,station,timestamp,air_temp_f,dew_point_temp_f,relative_humidity,wind_speed_kts,wind_gust_kts,visibility_miles,sky_l1_coverage,sky_l2_coverage,sky_l3_coverage,present_weather_codes,1hr_precipitation_inches,pressure_altimeter_inches
0,HTS,2025-01-01 00:51:00,46.00,37.00,70.64,15.00,25.00,10.00,FEW,BKN,OVC,M,0.01,29.66
1,SAN,2025-01-01 00:51:00,57.00,49.00,74.55,4.00,M,9.00,BKN,M,M,M,0.00,30.05
2,SGF,2025-01-01 00:52:00,36.00,29.00,75.46,11.00,M,10.00,OVC,M,M,M,0.00,30.14
3,BTR,2025-01-01 00:53:00,62.00,39.00,42.56,4.00,M,10.00,CLR,M,M,M,0.00,30.03
4,HOU,2025-01-01 00:53:00,64.00,40.00,41.25,4.00,M,10.00,CLR,M,M,M,0.00,30.07


## Missing values

In [15]:
df_t2.isna().sum()

station                      0
timestamp                    0
air_temp_f                   0
dew_point_temp_f             0
relative_humidity            0
wind_speed_kts               0
wind_gust_kts                0
visibility_miles             0
sky_l1_coverage              0
sky_l2_coverage              0
sky_l3_coverage              0
present_weather_codes        0
1hr_precipitation_inches     0
pressure_altimeter_inches    0
dtype: int64

* There are no actual missing values other than `M`.

## Type corrections before export

In [17]:
df_t2['pressure_altimeter_inches'].apply(type).value_counts()

pressure_altimeter_inches
<class 'str'>      2629406
<class 'float'>      65536
Name: count, dtype: int64

In [18]:
df_t3 = type_corrections(df_t2, 'pressure_altimeter_inches')

In [19]:
df_t3['pressure_altimeter_inches'].apply(type).value_counts()

pressure_altimeter_inches
<class 'str'>    2694942
Name: count, dtype: int64

## Export dataset to data lake

In [ ]:
s3_key = "staging/Flight_Delay_Prediction_Datasets/weather_data/weather_data.parquet"
write_data_to_lake_parquet(df_t3, s3_key)